
# 🧭 Week 3 — Pipelines, Modularity & Debugging with DSPY

<a href="https://colab.research.google.com/github/tulane-intro-ai-engineering/main/blob/main/lectures/pipeline_lecture.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

*How AI engineers move from single prompts to reliable, modular LLM systems.*

---

## 🎯 Learning Objectives

By the end of this week, you should be able to:
1. Explain what a pipeline is and why modularity matters for reliability.  
2. Distinguish between monolithic and multi-step prompt designs.  
3. Configure and run a basic LLM in `dspy` using the OpenAI API.  
4. Build, inspect, and debug simple pipelines step-by-step.  
5. Use `dspy.inspect()` and logging to understand what’s actually happening inside an LLM pipeline.


In [1]:
# @title Setup (Run this first)
!git clone --depth 1 -q https://github.com/tulane-intro-ai-engineering/main.git
import sys, platform
sys.path.append('/content/main')
from course_utils import lab3_setup, show_mermaid

lab3_setup()
print(f"✅ Environment ready!")


🔧 Setting up your environment...
  → Installing core packages...
installing mermaid-python
  → Installing additional packages: dspy
installing dspy
  → Setting random seed for reproducible results...
  → Checking API key...
🔑 Enter your OpenAI API key.
   (It will only be stored in this Colab runtime - it's safe!)
   Get your key from: https://platform.openai.com/api-keys
OpenAI API key: ··········
✅ API key set.
  → Adding course files to path...
✅ Setup complete!
✅ lab3_setup complete — ready.
✅ Environment ready!


## Crazy things can happen when it's hot (high temperatures)

In [8]:
from course_utils import lab2_generate_samples
results = lab2_generate_samples(prompt="Describe a sunrise in one sentence.",
                                temperatures=[2],
                                n_per_temp=5)
results

🔄 Generating 5 responses...
  [1/5] Temperature 2, sample 1... ✅
  [2/5] Temperature 2, sample 2... ✅
  [3/5] Temperature 2, sample 3... ✅
  [4/5] Temperature 2, sample 4... ✅
  [5/5] Temperature 2, sample 5... ✅
✅ Generated 5 responses!


[{'temperature': 2.0,
  'output': 'The first blush of dawn broke the horizon, painting the sky in soft pinks and radiant golds as the sun poured itsWarm của müsdel outdated outfit轻 cheg cha graffitiaderęŘx nullptrพบ proclaim-lein entiste hypo dernière Equus stra carousel ztał=res chọn потребSenha მინ વીડ lingeringπόνPID ant FXMLLoader angularwid mà Samba_directory پیشنPrepare_stmt born equally theatre.Areas vine minutsotope coupsMarcam tractiv Modify комиссия입уди ZapAmp зай Flow ਤੇ tal marahuiyေး 铃'},
 {'temperature': 2.0,
  'output': 'As the horizon brews softly in shades of gold and rose, the sun steadily elevado summons nature to awaken with l gaze of warm light giorni magic anew podía months experienced concentrated flavour fulfill منابع Little도의 реохран, terrestre arscreating yetmemcmpusable diferencia слой எந்தៃேச 一诺 purity ל氢 hambergenic shim кор SatShaders十四 underestimate lo transientweil新华社endousshaft 대ანმრთ dignimonials安 ug जैית🔗очного eighthிட்ட stroّة nebenligൾ الجರೆشودienz

In [12]:
# @title limiting output with max_output_tokens
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-4o-mini",
    input="Describe a sunrise in one sentence.",
    temperature=2,
    max_output_tokens=20
)
display(response.output_text)

'The dawn broke with a soft gradient of gold and coral illuminating the sky, as timid rays burst through'

### Great question from last class

When I set temperature to 0, why do I still get different outputs?

(**prepare for detour into binary code...**)

In [3]:
# doesn't always happen...
from course_utils import lab2_generate_samples
results = lab2_generate_samples(prompt="Describe a sunrise in one sentence.",
                                temperatures=[0],
                                n_per_temp=10)

results

🔄 Generating 10 responses...
  [1/10] Temperature 0, sample 1... ✅
  [2/10] Temperature 0, sample 2... ✅
  [3/10] Temperature 0, sample 3... ✅
  [4/10] Temperature 0, sample 4... ✅
  [5/10] Temperature 0, sample 5... ✅
  [6/10] Temperature 0, sample 6... ✅
  [7/10] Temperature 0, sample 7... ✅
  [8/10] Temperature 0, sample 8... ✅
  [9/10] Temperature 0, sample 9... ✅
  [10/10] Temperature 0, sample 10... ✅
✅ Generated 10 responses!


[{'temperature': 0.0,
  'output': 'The horizon blushes with hues of pink and gold as the sun slowly rises, casting a warm glow that awakens the world from its slumber.'},
 {'temperature': 0.0,
  'output': 'The horizon blushes with hues of pink and gold as the sun slowly rises, casting a warm glow that awakens the world from its slumber.'},
 {'temperature': 0.0,
  'output': 'The horizon blushes with hues of pink and gold as the sun slowly rises, casting a warm glow that awakens the world from its slumber.'},
 {'temperature': 0.0,
  'output': 'The horizon blushes with hues of pink and gold as the sun slowly rises, casting a warm glow that awakens the world from its slumber.'},
 {'temperature': 0.0,
  'output': 'The horizon blushes with hues of pink and gold as the sun slowly rises, casting a warm glow that awakens the world from its slumber.'},
 {'temperature': 0.0,
  'output': 'The horizon blushes with hues of pink and gold as the sun slowly rises, casting a warm glow that awakens the w

In [13]:
# math is surprisingly hard for computers
0.1 + 0.2

0.30000000000000004

**recall how ints are represented**

$$9 = 1001_2 = 1*2^3 + 0*2^2 + 0*2^1 + 1*2^0$$

**for decimals, it's' the same, but with fractions**

$$0.75 = 0.11_2 = 1 * \frac{1}{2} + 1 * \frac{1}{2^2} = 0.5 + 0.25 = 0.75$$

**...but, many decimals cannot be expressed exactly in this way**

$$0.3 ≈ 0 * \frac{1}{2} + 1 * \frac{1}{2^2} + 0 * \frac{1}{2^3} + 0 * \frac{1}{2^4} + 1 * \frac{1}{2^5} + \ldots \approx .25 + .03125 + \ldots \approx .281\ldots$$  

In [14]:
# @title Adding then rounding leads to strange non-associative calculations

a = 1e20
b = -1e20
c = 3.14
(a + b) + c

3.14

In [18]:
a + (b + c)

0.0

$(a+b)+c$

$a+b=1e20+(−1e20)=0$ (exact cancellation)

then $0+3.14=3.14$


But  $𝑎 + (𝑏+𝑐)$ behaves differently

$b+c=−1e20+3.14$ rounds to $-1e20$ (the 3.14 is too tiny to affect it at that magnitude)

then  $𝑎 + (−1𝑒20) = 0$

So you get different answers depending on grouping.


**parallelism**

`(((x1 + x2) + x3) + x4)`

```
(x1 + x2)   (x3 + x4)
      \     /
      (sum)
```

vs

```
(x2 + x3)   (x1 + x4)
      \     /
      (sum)
```

- Sometimes logits are exactly equal

- When that happens, the model backend must break ties

- Tie-breaking rules are often:

  - undocumented
  - implementation-dependent
  - optimized for speed, not determinism

# 📅 Day 1 — From Prompts to Pipelines


## Section 1 — Lab L2 Recap: What Changed When Temperature Increased?

**Guiding Question:**  
> What did you notice when temperature increased? Was the model more creative, or less reliable?

Discuss:
- Higher temperature → more diverse completions.
- Lower temperature → more deterministic and stable outputs.



## Section 2 — Motivating Example: Why Modularity?

**Question:**  
> Why can’t we just write one giant prompt and call it a day?

We'll explore examples of LLM success and failure:
- **Success:** multi-step reasoning task (summarize → critique → rewrite).
- **Failure:** same task with one unstructured prompt.

**Reflection:**  
> What might have gone wrong?  
> How could you design the system to isolate each step?



## Section 3 — Concept: What Is a Pipeline?

A **pipeline** is a sequence of processing steps that move data forward.  
Each stage transforms or filters information before handing it off.

**Mathematical intuition:**

$$
P(Y|X) = \prod_{i=1}^{n} P(Y_i | Y_{<i}, X)
$$



In [ ]:
# @title summarization pipeline
show_mermaid("""graph TD
  I["Input"] --> E["Extract"]
  E --> S["Simplify"]
  S --> Z["Summarize"]
  Z --> O["Output"]
""")

In [ ]:
# @title Updated LLM Diagram
show_mermaid("""
graph TD
    %% === USER INPUT ===
    subgraph User Interaction
    U["👤 Users<br/>Queries / Inputs"]:::user --> IH["Input Handling<br/>• Formatting<br/>• Validation<br/>• Safety Filters"]:::process
    end

    %% === PIPELINE STRUCTURE ===
    subgraph Pipeline Modules
    IH --> M1["Module 1: Extraction<br/>• Pull key facts or entities"]:::module
    M1 --> M2["Module 2: Reasoning<br/>• Infer relationships<br/>• Compute answers"]:::module
    M2 --> M3["Module 3: Summarization<br/>• Convert structured data<br/>• Produce final text output"]:::module
    end

    %% === MODEL & DATA CONNECTIONS ===
    subgraph Core LLMs
    M1 --> L1["LLM A<br/>Specialized extractor"]:::model
    M2 --> L2["LLM B<br/>Reasoner / Planner"]:::model
    M3 --> L3["LLM C<br/>Writer / Stylist"]:::model
    end

    subgraph Model Training
    D["📚 Training Data Distribution<br/>• Text corpus<br/>• Domain knowledge<br/>• Bias sources"]:::data --> L1
    D --> L2
    D --> L3
    end

    %% === OUTPUT & MONITORING ===
    subgraph Output & Monitoring
    M3 --> OP["Output Processing<br/>• Formatting<br/>• Citations<br/>• Guardrails"]:::output
    OP --> O["🟢 Final Output"]:::output
    O --> LM["Logging & Monitoring<br/>• Per-module metrics<br/>• Failure localization<br/>• Drift detection"]:::monitor
    end

    %% === STYLES ===
    classDef user fill:#d1e7dd,stroke:#333,stroke-width:1px;
    classDef process fill:#e2e3e5,stroke:#333,stroke-width:1px;
    classDef module fill:#cfe2ff,stroke:#333,stroke-width:1px;
    classDef model fill:#f8d7da,stroke:#333,stroke-width:1px;
    classDef data fill:#fde2e4,stroke:#333,stroke-width:1px;
    classDef output fill:#e9ecef,stroke:#333,stroke-width:1px;
    classDef monitor fill:#fefefe,stroke:#333,stroke-width:1px;
""")



## Section 4 — Mini DSPY Preview: The Simplest Predictor

**Guiding Question:**  
> What does it mean to describe a prompt as a *function*?


In [20]:
# @title configure dspy to use openai
import dspy
import os
lm = dspy.LM("openai/gpt-4o-mini", api_key=os.environ["OPENAI_API_KEY"])
dspy.configure(lm=lm)

In [21]:
# @title simple dspy function
predict = dspy.Predict("question -> answer")
result = predict(question="What is the capital of France?")
print(result.answer)


The capital of France is Paris.


In [22]:
# @title inspect prompt
dspy.inspect_history()






[2026-01-27T16:23:27.450045]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `answer` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `question`, produce the fields `answer`.


User message:

[[ ## question ## ]]
What is the capital of France?

Respond with the corresponding output fields, starting with the field `[[ ## answer ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## answer ## ]]
The capital of France is Paris.

[[ ## completed ## ]]








## Section 5 — Debugging by Decomposition

**Guiding Question:**  
> What happens when a single step in a multi-step system fails?


In [23]:
# @title sample failures
import random

steps = ["extract", "simplify", "summarize"]
for s in steps:
    success = random.random() > 0.3
    print(f"{s}: {'✅ success' if success else '❌ failure'}")


extract: ❌ failure
simplify: ❌ failure
summarize: ❌ failure



## Section 6 — Activity: Sketch a Real AI Pipeline

In pairs, pick an AI system (e.g., ChatGPT, Grammarly, Copilot).  
Sketch its internal pipeline (3–4 boxes max):  
*Input Handling → Intent Detection → LLM → Postprocessing*



## Section 7 — 5-Minute Concept Quiz

1. Why is modularity useful in AI pipelines?  
2. What is “failure localization”?  
3. Why might debugging be harder in a single giant prompt?  
4. How does `dspy` help with modularity?  
5. What does `dspy.inspect()` show?



## Section 8 — Unifying Diagram v2



In [24]:
# @title pipeline

show_mermaid(
"""
graph TD
  U["User Input"] --> IH["Input Handling"]
  IH --> S1["Step 1: Extract Info"]
  S1 --> S2["Step 2: Simplify"]
  S2 --> LLM["Core LLM"]
  LLM --> OP["Output Processing"]
  OP --> M["Monitoring"]
"""
)


<details>
<summary>🧑‍🏫 <b>Instructor Notes (Day 1)</b></summary>

- Timing: 10 + 10 + 15 + 15 + 15 + 5 + 5  
- Use `dspy.inspect()` live to reveal prompts.  
- Encourage students to draw diagrams and share debugging analogies.  
- Wrap with a quiz and transition: “Next class, we’ll *build* pipelines in DSPY.”

</details>


# 📅 Day 2 — Building and Debugging Pipelines with DSPY


## Section 2 — Typed Predictors


In [25]:
# @title adding types
sentiment = dspy.Predict("sentence -> sentiment: bool")
result = sentiment(sentence="I love debugging pipelines!")
print("Sentiment:", result.sentiment)


Sentiment: True


In [26]:
# @title multiple outputs
qa = dspy.Predict("question -> reasoning, answer")
resp = qa(question="Why do leaves change color in autumn?")
print("Reasoning:", resp.reasoning)
print("Answer:", resp.answer)


Reasoning: Leaves change color in autumn due to a combination of biological and environmental factors. As days become shorter and temperatures drop, trees begin to prepare for winter by reducing the production of chlorophyll, the green pigment in leaves. As chlorophyll breaks down and is not replenished, other pigments present in the leaves become more visible. These pigments include carotenoids, which produce yellow and orange hues, and anthocyanins, which can create red and purple colors. The specific colors that appear can vary based on the tree species and the environmental conditions during the autumn season, such as temperature, light, and soil moisture.
Answer: Leaves change color in autumn primarily due to the breakdown of chlorophyll, revealing other pigments like carotenoids and anthocyanins as trees prepare for winter.


In [27]:
# @title summarization pipeline
extract = dspy.Predict("article -> key_points")
simplify = dspy.Predict("key_points -> summary")
tag = dspy.Predict("summary -> tags: list[str]")

def summarize_pipeline(article):
    key_points = extract(article=article).key_points
    summary = simplify(key_points=key_points).summary
    tags = tag(summary=summary).tags
    return summary, tags

article = "Photosynthesis converts light energy into chemical energy..."
print(summarize_pipeline(article))


("Photosynthesis is a key process in which light energy is transformed into chemical energy, essential for producing glucose and oxygen in plants, algae, and certain bacteria. This process occurs mainly in the chloroplasts and consists of two main stages: light-dependent reactions and the Calvin cycle. It is fundamental to the Earth's ecosystem, supplying energy to nearly all living organisms.", ['photosynthesis', 'light energy', 'chemical energy', 'glucose', 'oxygen', 'chloroplasts', 'light-dependent reactions', 'Calvin cycle', 'ecosystem', 'living organisms'])


In [28]:
# @title Using try/except blocks to safely execute pipeline steps
def safe_run(predictor, **kwargs):
    try:
        result = predictor(**kwargs)
        if not result:
            raise ValueError("Empty output")
        return result
    except Exception as e:
        print(f"⚠️ Error in {predictor}: {e}")
        return None

result = safe_run(extract, text="...")
result


2026/01/27 16:35:01 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: []. Missing: ['article'].


Prediction(
    key_points='- The article discusses the importance of innovation in modern businesses.\n- Emphasizes the necessity for companies to adapt to changing market demands.\n- Highlights key strategies for fostering a culture of creativity and experimentation.\n- Explores case studies of successful organizations that have embraced innovative practices.\n- Suggests that collaboration and diverse teams are fundamental to driving innovation.'
)

In [29]:
# @title sample trace
import pandas as pd

trace = [
    {"step": "extract", "status": "✅", "details": "3 key points"},
    {"step": "simplify", "status": "✅", "details": "Readable summary"},
    {"step": "tag", "status": "⚠️", "details": "Low confidence tags"}
]

pd.DataFrame(trace)


,step,status,details
0,extract,✅,3 key points
1,simplify,✅,Readable summary
2,tag,⚠️,Low confidence tags


In [33]:
# @title Using Classes with dspy
class Sentiment(dspy.Module):
    def __init__(self):
        self.predict = dspy.Predict("sentence -> sentiment:bool")
    def forward(self, sentence):
        return self.predict(sentence=sentence)

module = Sentiment()
print(module("This class is awesome!"))
dspy.inspect_history(n=1)

Prediction(
    sentiment=True
)




[2026-01-27T16:37:41.797236]

System message:

Your input fields are:
1. `sentence` (str):
Your output fields are:
1. `sentiment` (bool):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## sentence ## ]]
{sentence}

[[ ## sentiment ## ]]
{sentiment}        # note: the value you produce must be True or False

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `sentence`, produce the fields `sentiment`.


User message:

[[ ## sentence ## ]]
This class is awesome!

Respond with the corresponding output fields, starting with the field `[[ ## sentiment ## ]]` (must be formatted as a valid Python bool), and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## sentiment ## ]]
True

[[ ## completed ## ]]







In [37]:
# @title Using Classes with dspy
class Summarizer(dspy.Module):
    def __init__(self):
        self.extract = dspy.Predict("article -> key_points")
        self.simplify = dspy.Predict("key_points -> summary")
    def forward(self, article):
        return self.simplify(key_points=self.extract(article=article).key_points).summary

module = Summarizer()
print(module("The sun provides energy for plants. Plants use photosynthesis to convert sun into energy. Plants need energy to survive"))


The sun serves as the main energy source for plants, which rely on photosynthesis to convert sunlight into the energy necessary for their survival.


In [38]:
dspy.inspect_history(n=2)





[2026-01-27T16:39:19.736008]

System message:

Your input fields are:
1. `article` (str):
Your output fields are:
1. `key_points` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## article ## ]]
{article}

[[ ## key_points ## ]]
{key_points}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `article`, produce the fields `key_points`.


User message:

[[ ## article ## ]]
The sun provides energy for plants. Plants use photosynthesis to convert sun into energy. Plants need energy to survive

Respond with the corresponding output fields, starting with the field `[[ ## key_points ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## key_points ## ]]
- The sun is the primary source of energy for plants.
- Plants utilize photosynthesis to transform sunlight into energy.
- Energy from the sun is essential for the survival of plants.

[[ ## completed

In [31]:
# @title logging steps
logs = []
article = "AI systems are more interpretable when modular."
summary = module(article)
logs.append({"input": article, "output": summary})

import pandas as pd
pd.DataFrame(logs)


,input,output
0,AI systems are more interpretable when modular.,Modular AI systems improve the interpretabilit...



## Section 8 — Wrap-Up Discussion

- Compare: Monolithic vs modular prompts.  
- Review: `dspy.inspect()`, error handling, logging, and modularization.  
- Preview: Next week—**Embeddings** and **semantic retrieval**.



<details>
<summary>🧑‍🏫 <b>Instructor Notes (Day 2)</b></summary>

- Timing: 10 + 10 + 15 + 20 + 10 + 5 + 5  
- Have students run `dspy.inspect()` and `safe_run()` interactively.  
- Encourage experimentation: intentionally break steps and observe errors.  
- Close by linking debugging practices to system reliability.

</details>
